In [1]:
import cv2
import pytesseract
import re
import numpy as np


In [2]:
def extract_text_from_image(image_path):
    img = cv2.imread(image_path)

    if img is None:
        raise ValueError("Invalid image path")

    img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 11, 17, 17)

    thresh = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31, 2
    )

    config = r'--oem 3 --psm 6'
    return pytesseract.image_to_string(thresh, config=config)


In [3]:
def parse_nutrients(text):
    text = text.lower()

    patterns = {
        "energy_100g": r"energy[^\d]*([\d]+\.?[\d]*)",
        "fat_100g": r"(?:total fat|fat)[^\d]*([\d]+\.?[\d]*)",
        "saturated_fat_100g": r"saturated fat[^\d]*([\d]+\.?[\d]*)",
        "carbohydrates_100g": r"carbohydrate[^\d]*([\d]+\.?[\d]*)",
        "sugars_100g": r"(?:total sugars|added sugars|sugars)[^\d]*([\d]+\.?[\d]*)",
        "proteins_100g": r"protein[^\d]*([\d]+\.?[\d]*)",
        "salt_100g": r"salt[^\d]*([\d]+\.?[\d]*)",
        "sodium_100g": r"sodium[^\d]*([\d]+\.?[\d]*)"
    }

    nutrients = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text)
        nutrients[key] = float(match.group(1)) if match else 0.0

    # Sodium → salt fallback
    if nutrients["salt_100g"] == 0 and nutrients["sodium_100g"] > 0:
        nutrients["salt_100g"] = (nutrients["sodium_100g"] / 1000) * 2.5

    return nutrients


In [4]:
def infer_additives_n(text):
    additive_keywords = [
        "emulsifier", "stabilizer", "preservative",
        "flavour", "flavor", "colour", "color",
        "sweetener", "enzyme", "thickener"
    ]

    count = sum(1 for w in additive_keywords if w in text)
    if "ingredients" in text and count == 0:
        return 1
    return count


def infer_nova_group(text):
    ultra = [
        "flavour", "emulsifier", "stabilizer", "sweetener",
        "enzyme", "isolated", "protein powder", "instant",
        "fortified", "artificial"
    ]
    processed = ["cheese", "bread", "butter", "salted", "roasted"]
    natural = ["fruit", "vegetable", "grain", "milk", "egg", "rice"]

    if any(k in text for k in ultra):
        return 4
    elif any(k in text for k in processed):
        return 3
    elif any(k in text for k in natural):
        return 1
    return 4


In [5]:
FEATURES = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]


In [ ]:
import numpy as np

def predict_food_health(model, food_dict):
    """
    Predict health label and confidence using trained Spark Logistic Regression
    """

    # Extract coefficients
    coef = model.coefficients.toArray()
    intercept = model.intercept

    # Build feature vector in correct order
    X = np.array([food_dict[f] for f in FEATURES])

    # Logistic regression equation
    z = np.dot(coef, X) + intercept
    prob_healthy = 1 / (1 + np.exp(-z))

    pred = 1 if prob_healthy >= 0.5 else 0
    return pred, [1 - prob_healthy, prob_healthy]


In [ ]:
 image_path = r"D:\CDAC PROJECT\FoodNutritionAnalyzer\Screenshot 2026-01-26 152844.png"


text = extract_text_from_image(image_path)
print("📝 OCR TEXT:\n", text)

nutrients = parse_nutrients(text)

nutrients["additives_n"] = infer_additives_n(text)
nutrients["nova_group"] = infer_nova_group(text)

sample_food = {
    "energy-kcal_100g": nutrients["energy_100g"],
    "fat_100g": nutrients["fat_100g"],
    "saturated-fat_100g": nutrients["saturated_fat_100g"],
    "carbohydrates-total_100g": nutrients["carbohydrates_100g"],
    "sugars_100g": nutrients["sugars_100g"],
    "fiber_100g": nutrients.get("fiber_100g", 0.0),
    "proteins_100g": nutrients["proteins_100g"],
    "salt_100g": nutrients["salt_100g"],
    "sodium_100g": nutrients["sodium_100g"] / 1000,
    "additives_n": nutrients["additives_n"],
    "nova_group": nutrients["nova_group"]
}

pred, prob = predict_food_health(model, sample_food)

label = "Healthy" if pred == 1 else "Unhealthy"

print("Prediction:", label)
print("Probability (Unhealthy, Healthy):", prob)
